# 自定义坐标 PDE 的物理与混合监督

**路径：扩展 Extension。** 定义坐标 PDE，并在同一训练入口比较 DeepONet 的 `physics_only` 与 `hybrid`。

本页运行的是固定随机种子、CPU、小步数 **smoke**，只验证完整训练与结果链路，不是精度结论。
浏览器中的方法配置器仍是并行入口：它下载可训练 `.py`，不会在浏览器训练或上传数据。
notebook 源文件位于 `examples/tutorials/`，可下载后在仓库根目录重新执行。


In [1]:
from pathlib import Path
import json
import sys

import openpinn as op

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("OpenPINN", op.__version__, "| repo", ROOT.name)


OpenPINN 0.2.0 | repo OpenPINN-upstream


In [2]:
import math
import torch

problem = op.PDEProblem(
    op.Interval(-1.0, 1.0),
    lambda x, u: op.hessian(u, x, i=0, j=0) + (math.pi**2) * u,
    [
        op.DirichletBC(0.0, boundary=lambda x: torch.isclose(x[:, 0], x.new_tensor(-1.0))),
        op.DirichletBC(0.0, boundary=lambda x: torch.isclose(x[:, 0], x.new_tensor(1.0))),
    ],
    name="tutorial_helmholtz",
)
coordinates = torch.linspace(-1.0, 1.0, 9).reshape(-1, 1)
observation = op.OperatorObservation(
    coordinates=coordinates,
    values=torch.sin(math.pi * coordinates),
    source=Path("tutorial_observation.csv"),
)
print(problem.name, "| observation points", len(coordinates))


tutorial_helmholtz | observation points 9


In [3]:
common = dict(
    n_dims=1, n_fields=1, n_sensors=8, latent_dim=4,
    branch_hidden=[8], trunk_hidden=[8], batch_size=2,
    n_query=8, n_modes=2, steps=2, seed=0, lr=1e-2,
)
summaries = {}
for mode in ("physics_only", "hybrid"):
    config = op.build_operator_config("deeponet", training_mode=mode, **common)
    result = op.train_operator_problem(
        config, problem,
        observation=observation if mode == "hybrid" else None,
        device="cpu",
    )
    summaries[mode] = {
        "final_loss": result.final_loss,
        "final_physics_loss": result.metrics["final_physics_loss"],
        "final_data_loss": result.metrics["final_data_loss"],
    }
print(json.dumps(summaries, indent=2))


{
  "physics_only": {
    "final_loss": 0.0,
    "final_physics_loss": 0.0,
    "final_data_loss": 0.0
  },
  "hybrid": {
    "final_loss": 0.45464181900024414,
    "final_physics_loss": 0.026904411613941193,
    "final_data_loss": 0.42773741483688354
  }
}
